# Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import os
from tqdm import tqdm
import copy
import matplotlib.pyplot as plt

WORD_SIZE = 16
MASK_VAL = 2 ** WORD_SIZE - 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Running on {DEVICE}.")

Running on cuda.


# Data generation

In [2]:
def rol(x, r):
    return ((x << r) & MASK_VAL) | (x >> (WORD_SIZE - r))


def ror(x, r):
    return ((x >> r) | (x << (WORD_SIZE - r))) & MASK_VAL


def enc_one_round(p, k):
    c0, c1 = p[0], p[1]
    c0 = ror(c0, 7)
    c0 = (c0 + c1) & MASK_VAL
    c0 = c0 ^ k
    c1 = rol(c1, 2)
    c1 = c1 ^ c0
    return c0, c1


def dec_one_round_vec(c0, c1, k):
    c1 = c1 ^ c0
    c1 = ((c1 >> 2) | (c1 << 14)) & MASK_VAL
    c0 = c0 ^ k
    c0 = (c0 - c1) & MASK_VAL
    c0 = ((c0 << 7) | (c0 >> 9)) & MASK_VAL
    return c0, c1


def extract_features_2024(c0l, c0r, c1l, c1r):
    f1 = c0l ^ c1l
    f2 = c0l ^ c0r
    f3 = c0l ^ c1r
    n_samples = c0l.shape[0] if isinstance(c0l, np.ndarray) else c0l.size(0)
    X_bin = np.zeros((n_samples, 3, 16), dtype=np.float32)
    for i in range(16):
        bit_mask = 1 << i
        X_bin[:, 0, i] = (f1 & bit_mask) >> i
        X_bin[:, 1, i] = (f2 & bit_mask) >> i
        X_bin[:, 2, i] = (f3 & bit_mask) >> i
    return X_bin


def make_data_hard_negative(n_samples, rounds, diff=(0x0040, 0x0000)):
    keys = np.frombuffer(os.urandom(n_samples * 8), dtype=np.uint16).reshape(n_samples, 4)
    p0l = np.frombuffer(os.urandom(n_samples * 2), dtype=np.uint16)
    p0r = np.frombuffer(os.urandom(n_samples * 2), dtype=np.uint16)
    p1l = p0l ^ diff[0]
    p1r = p0r ^ diff[1]

    ks = np.zeros((n_samples, rounds + 1), dtype=np.uint16)
    l = [keys[:, 0], keys[:, 1], keys[:, 2]]
    ks[:, 0] = keys[:, 3]
    for i in range(rounds):
        l[i % 3] = ((l[i % 3] >> 7) | (l[i % 3] << 9)) & MASK_VAL
        l[i % 3] = (l[i % 3] + ks[:, i]) & MASK_VAL
        l[i % 3] = l[i % 3] ^ i
        ks[:, i + 1] = ((ks[:, i] << 2) | (ks[:, i] >> 14)) & MASK_VAL
        ks[:, i + 1] = ks[:, i + 1] ^ l[i % 3]

    c0l, c0r = p0l.copy(), p0r.copy()
    c1l, c1r = p1l.copy(), p1r.copy()

    for i in range(rounds):
        ki = ks[:, i]
        c0l = ((c0l >> 7) | (c0l << 9)) & MASK_VAL
        c0l = (c0l + c0r) & MASK_VAL
        c0l ^= ki
        c0r = ((c0r << 2) | (c0r >> 14)) & MASK_VAL
        c0r ^= c0l
        c1l = ((c1l >> 7) | (c1l << 9)) & MASK_VAL
        c1l = (c1l + c1r) & MASK_VAL
        c1l ^= ki
        c1r = ((c1r << 2) | (c1r >> 14)) & MASK_VAL
        c1r ^= c1l

    Y = np.frombuffer(os.urandom(n_samples), dtype=np.uint8) & 1
    rand_mask = (Y == 0)
    n_rand = np.sum(rand_mask)

    if n_rand > 0:
        c0l_neg = c0l[rand_mask]
        c0r_neg = c0r[rand_mask]
        c1l_neg = c1l[rand_mask]
        c1r_neg = c1r[rand_mask]

        k_true_neg = ks[rand_mask, rounds]

        c0l_next = ((c0l_neg >> 7) | (c0l_neg << 9)) & MASK_VAL
        c0l_next = (c0l_next + c0r_neg) & MASK_VAL
        c0l_next ^= k_true_neg
        c0r_next = ((c0r_neg << 2) | (c0r_neg >> 14)) & MASK_VAL
        c0r_next ^= c0l_next

        c1l_next = ((c1l_neg >> 7) | (c1l_neg << 9)) & MASK_VAL
        c1l_next = (c1l_next + c1r_neg) & MASK_VAL
        c1l_next ^= k_true_neg
        c1r_next = ((c1r_neg << 2) | (c1r_neg >> 14)) & MASK_VAL
        c1r_next ^= c1l_next

        wrong_keys = np.frombuffer(os.urandom(n_rand * 2), dtype=np.uint16).copy()
        wrong_keys[wrong_keys == k_true_neg] ^= 1

        c0l_wrong, c0r_wrong = dec_one_round_vec(c0l_next, c0r_next, wrong_keys)
        c1l_wrong, c1r_wrong = dec_one_round_vec(c1l_next, c1r_next, wrong_keys)

        c0l[rand_mask] = c0l_wrong
        c0r[rand_mask] = c0r_wrong
        c1l[rand_mask] = c1l_wrong
        c1r[rand_mask] = c1r_wrong

    X = extract_features_2024(c0l, c0r, c1l, c1r)
    return torch.tensor(X), torch.tensor(Y, dtype=torch.float32)


def make_data_standard(n_samples, rounds, diff=(0x0040, 0x0000)):
    keys = np.frombuffer(os.urandom(n_samples * 8), dtype=np.uint16).reshape(n_samples, 4)
    p0l = np.frombuffer(os.urandom(n_samples * 2), dtype=np.uint16)
    p0r = np.frombuffer(os.urandom(n_samples * 2), dtype=np.uint16)
    p1l = p0l ^ diff[0]
    p1r = p0r ^ diff[1]

    ks = np.zeros((n_samples, rounds), dtype=np.uint16)
    l = [keys[:, 0], keys[:, 1], keys[:, 2]]
    ks[:, 0] = keys[:, 3]
    for i in range(rounds - 1):
        l[i % 3] = ((l[i % 3] >> 7) | (l[i % 3] << 9)) & MASK_VAL
        l[i % 3] = (l[i % 3] + ks[:, i]) & MASK_VAL
        l[i % 3] = l[i % 3] ^ i
        ks[:, i + 1] = ((ks[:, i] << 2) | (ks[:, i] >> 14)) & MASK_VAL
        ks[:, i + 1] = ks[:, i + 1] ^ l[i % 3]

    c0l, c0r = p0l.copy(), p0r.copy()
    c1l, c1r = p1l.copy(), p1r.copy()

    for i in range(rounds):
        ki = ks[:, i]
        c0l = ((c0l >> 7) | (c0l << 9)) & MASK_VAL
        c0l = (c0l + c0r) & MASK_VAL
        c0l ^= ki
        c0r = ((c0r << 2) | (c0r >> 14)) & MASK_VAL
        c0r ^= c0l
        c1l = ((c1l >> 7) | (c1l << 9)) & MASK_VAL
        c1l = (c1l + c1r) & MASK_VAL
        c1l ^= ki
        c1r = ((c1r << 2) | (c1r >> 14)) & MASK_VAL
        c1r ^= c1l

    Y = np.frombuffer(os.urandom(n_samples), dtype=np.uint8) & 1
    rand_mask = (Y == 0)
    n_rand = np.sum(rand_mask)

    c1l[rand_mask] = np.frombuffer(os.urandom(n_rand * 2), dtype=np.uint16)
    c1r[rand_mask] = np.frombuffer(os.urandom(n_rand * 2), dtype=np.uint16)

    X = extract_features_2024(c0l, c0r, c1l, c1r)
    return torch.tensor(X), torch.tensor(Y, dtype=torch.float32)


# Model definition

In [3]:
class ResBlock(nn.Module):
    def __init__(self, channels, kernel_size=3):
        super(ResBlock, self).__init__()
        self.conv1 = nn.Conv1d(channels, 32, kernel_size, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, channels, kernel_size, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        res = x
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = x + res
        return x


class ImprovedCNN(nn.Module):
    def __init__(self, depth=10):
        super(ImprovedCNN, self).__init__()
        self.in_channels = 3
        self.res_tower = nn.Sequential(*[ResBlock(self.in_channels) for _ in range(depth)])
        self.fc1 = nn.Linear(3 * 16, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(64, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.out = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.res_tower(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.sigmoid(self.out(x))
        return x


# Train

In [4]:
def finetune_dnd(model_filename, rounds, depth, n_epochs=20, batch_size=5000, lr=5e-6, load_dir="light_models", save_dir="light_models"):
    load_path = os.path.join(load_dir, model_filename)
    save_path = os.path.join(save_dir, f"finetuned_hn_{model_filename}")

    model = ImprovedCNN(depth=depth).to(DEVICE)
    if os.path.exists(load_path):
        checkpoint = torch.load(load_path, map_location=DEVICE, weights_only=True)
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        raise FileNotFoundError(f"Model file {load_path} not found.")

    loss_fn = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0)

    TOTAL_SAMPLES = 10 ** 7
    VAL_SIZE = 10 ** 6
    TRAIN_SIZE = TOTAL_SAMPLES - VAL_SIZE
    steps_per_epoch = TRAIN_SIZE // batch_size

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, min_lr=1e-7)

    print(f"\nFine-Tuning DND {model_filename} (Rounds: {rounds}, Depth: {depth})")
    print("-" * 60)

    best_val_acc = 0.0
    best_model_state = copy.deepcopy(model.state_dict())

    for epoch in range(n_epochs):
        X_all, Y_all = make_data_hard_negative(TOTAL_SAMPLES, rounds)
        X_train, X_val = X_all[:TRAIN_SIZE], X_all[TRAIN_SIZE:]
        Y_train, Y_val = Y_all[:TRAIN_SIZE], Y_all[TRAIN_SIZE:]

        train_ds = TensorDataset(X_train.to(DEVICE), Y_train.to(DEVICE))
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

        model.train()
        epoch_train_loss = 0.0

        iter_wrapper = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{n_epochs}")

        for xb, yb in iter_wrapper:
            optimizer.zero_grad()
            out = model(xb).squeeze()
            loss = loss_fn(out, yb)
            loss.backward()
            optimizer.step()

            epoch_train_loss += loss.item()

        avg_train_loss = epoch_train_loss / len(train_loader)

        # Validation Phase
        model.eval()
        with torch.no_grad():
            val_ds = TensorDataset(X_val.to(DEVICE), Y_val.to(DEVICE))
            val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
            val_loss_sum, val_acc_sum = 0.0, 0.0
            for xb, yb in val_loader:
                out = model(xb).squeeze()
                loss = loss_fn(out, yb)
                val_loss_sum += loss.item()
                acc = ((out > 0.5) == (yb > 0.5)).float().mean().item()
                val_acc_sum += acc

            avg_val_loss = val_loss_sum / len(val_loader)
            avg_val_acc = val_acc_sum / len(val_loader)
            scheduler.step(avg_val_acc)

        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            best_model_state = copy.deepcopy(model.state_dict())
            torch.save({
                "rounds": rounds,
                "depth": depth,
                "epoch": epoch + 1,
                "best_val_acc": best_val_acc,
                "model_state_dict": best_model_state,
            }, save_path)

        current_lr = optimizer.param_groups[0]['lr']
        print(f"  Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f} | Val Acc: {avg_val_acc:.4f} | LR: {current_lr:.2e}")

    # Restore the best performing weights before returning
    model.load_state_dict(best_model_state)
    return model


# =============================================================================
# Fine-Tuning Pipeline: Batch-based (For 8r)
# =============================================================================

def finetune_N8(model_filename, rounds=8, depth=1, lr=5e-6, load_dir="light_models", save_dir="light_models"):
    load_path = os.path.join(load_dir, model_filename)
    save_path = os.path.join(save_dir, f"finetuned_hn_{model_filename}")

    model = ImprovedCNN(depth=depth).to(DEVICE)
    if os.path.exists(load_path):
        checkpoint = torch.load(load_path, map_location=DEVICE, weights_only=True)
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        raise FileNotFoundError(f"Model file {load_path} not found.")

    loss_fn = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0)

    n_batches_total = (10 ** 8) // 10000  # 10,000 batches
    batch_size = 10000

    # Split into 10 evaluation checkpoints (1,000 batches each)
    eval_checkpoints = 10
    batches_per_checkpoint = n_batches_total // eval_checkpoints

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, min_lr=1e-7)

    print(f"\nFine-Tuning N8 {model_filename} (Rounds: {rounds}, Depth: {depth})")
    print("-" * 60)

    # Generate a static validation set for consistent evaluation
    print("Generating validation set...")
    X_val, Y_val = make_data_hard_negative(10**6, rounds, diff=(0x0040, 0x0000))
    val_ds = TensorDataset(X_val.to(DEVICE), Y_val.to(DEVICE))
    val_loader = DataLoader(val_ds, batch_size=5000, shuffle=False)

    best_val_acc = 0.0
    best_model_state = copy.deepcopy(model.state_dict())

    for checkpoint_idx in range(eval_checkpoints):
        model.train()
        total_loss = 0.0

        desc = f"Checkpoint {checkpoint_idx + 1}/{eval_checkpoints}"
        pbar = tqdm(range(batches_per_checkpoint), desc=desc)

        for _ in pbar:
            X, Y = make_data_hard_negative(batch_size, rounds, diff=(0x0040, 0x0000))
            X, Y = X.to(DEVICE), Y.to(DEVICE)

            optimizer.zero_grad()
            out = model(X).squeeze()
            loss = loss_fn(out, Y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            current_lr = optimizer.param_groups[0]["lr"]
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{current_lr:.1e}")

        avg_train_loss = total_loss / batches_per_checkpoint

        # Evaluation Phase
        model.eval()
        val_loss_sum, val_acc_sum = 0.0, 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                out = model(xb).squeeze()
                loss = loss_fn(out, yb)
                val_loss_sum += loss.item()
                acc = ((out > 0.5) == (yb > 0.5)).float().mean().item()
                val_acc_sum += acc

        avg_val_loss = val_loss_sum / len(val_loader)
        avg_val_acc = val_acc_sum / len(val_loader)
        scheduler.step(avg_val_acc)
        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            best_model_state = copy.deepcopy(model.state_dict())
            torch.save({
                "rounds": rounds,
                "depth": depth,
                "best_val_acc": best_val_acc,
                "model_state_dict": best_model_state,
            }, save_path)

        print(f"  Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f} | Val Acc: {avg_val_acc:.4f} | LR: {current_lr:.2e}")

    # Restore the best performing weights before returning
    model.load_state_dict(best_model_state)
    return model

# Evaluation

In [5]:
def evaluate_model(model_path, rounds, depth=10, n_samples=10 ** 6, batch_size=5000):
    model = ImprovedCNN(depth=depth).to(DEVICE)
    checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    loss_fn = nn.MSELoss()

    print(f"\nEvaluating: {os.path.basename(model_path)}")
    print(f"Configuration: {rounds} Rounds, Depth {depth}, {n_samples} Samples")
    print("-" * 60)

    # 1. Evaluate on Standard Distribution
    X_std, Y_std = make_data_standard(n_samples, rounds)
    std_loader = DataLoader(TensorDataset(X_std.to(DEVICE), Y_std.to(DEVICE)), batch_size=batch_size, shuffle=False)

    std_loss_sum, std_acc_sum = 0.0, 0.0
    with torch.no_grad():
        for xb, yb in tqdm(std_loader, desc="Standard Dist", leave=False):
            out = model(xb).squeeze()
            loss = loss_fn(out, yb)
            std_loss_sum += loss.item()
            acc = ((out > 0.5) == (yb > 0.5)).float().mean().item()
            std_acc_sum += acc

    avg_std_loss = std_loss_sum / len(std_loader)
    avg_std_acc = std_acc_sum / len(std_loader)

    # 2. Evaluate on Hard Negative Distribution
    X_hn, Y_hn = make_data_hard_negative(n_samples, rounds)
    hn_loader = DataLoader(TensorDataset(X_hn.to(DEVICE), Y_hn.to(DEVICE)), batch_size=batch_size, shuffle=False)

    hn_loss_sum, hn_acc_sum = 0.0, 0.0
    with torch.no_grad():
        for xb, yb in tqdm(hn_loader, desc="Hard Negative", leave=False):
            out = model(xb).squeeze()
            loss = loss_fn(out, yb)
            hn_loss_sum += loss.item()
            acc = ((out > 0.5) == (yb > 0.5)).float().mean().item()
            hn_acc_sum += acc

    avg_hn_loss = hn_loss_sum / len(hn_loader)
    avg_hn_acc = hn_acc_sum / len(hn_loader)

    print(f"  Standard (Random)    - Acc: {avg_std_acc:.4f} | Loss: {avg_std_loss:.5f}")
    print(f"  Hard Negative (Key)  - Acc: {avg_hn_acc:.4f} | Loss: {avg_hn_loss:.5f}")


def fine_tune_and_evaluate_all(directory="light_models", n_epochs=20, lr=1e-4):
    if not os.path.exists(directory):
        print(f"Directory {directory} not found.")
        return

    model_files = [f for f in os.listdir(directory) if f.endswith('.pt') and not f.startswith('finetuned_')]
    model_files.sort()

    for model_filename in model_files:
        load_path = os.path.join(directory, model_filename)
        checkpoint = torch.load(load_path, map_location=DEVICE, weights_only=True)

        rounds = checkpoint.get("rounds")
        depth = checkpoint.get("depth")

        # Fallback to the requested structure if depth is missing in models
        if depth is None:
            depth = 10 if rounds in [5, 6] else 1

        if rounds in [5, 6, 7]:
            finetune_dnd(model_filename, rounds, depth, n_epochs=n_epochs, lr=lr, load_dir=directory,
                         save_dir=directory)
        elif rounds == 8:
            finetune_N8(model_filename, rounds, depth, lr=lr, load_dir=directory, save_dir=directory)

    print("\n" + "=" * 60)
    print("Fine-tuning complete. Proceeding to evaluation phase.")
    print("=" * 60)

    # Evaluate both base models and newly fine-tuned models
    all_models = [f for f in os.listdir(directory) if f.endswith('.pt')]
    all_models.sort()

    for model_filename in all_models:
        model_path = os.path.join(directory, model_filename)
        checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=True)
        rounds = checkpoint.get("rounds")
        depth = checkpoint.get("depth")
        if depth is None:
            depth = 10 if rounds in [5, 6] else 1

        if rounds is not None:
            evaluate_model(model_path, rounds=rounds, depth=depth, n_samples=10 ** 6, batch_size=5000)



# Run fine-tuning

In [6]:
fine_tune_and_evaluate_all(directory="light_models", n_epochs=30, lr=5e-6)


Fine-Tuning DND dnd_speck32_r5_d10.pt (Rounds: 5, Depth: 10)
------------------------------------------------------------


Epoch 1/30: 100%|██████████| 1800/1800 [02:53<00:00, 10.37it/s]


  Train Loss: 0.06657 | Val Loss: 0.06487 | Val Acc: 0.9163 | LR: 5.00e-06


Epoch 2/30: 100%|██████████| 1800/1800 [03:04<00:00,  9.74it/s]


  Train Loss: 0.06448 | Val Loss: 0.06396 | Val Acc: 0.9172 | LR: 5.00e-06


...

  Standard (Random)    - Acc: 0.5003 | Loss: 0.25000
  Hard Negative (Key)  - Acc: 0.5014 | Loss: 0.25000

Evaluating: finetuned_hn_dnd_speck32_r5_d10.pt
Configuration: 5 Rounds, Depth 10, 1000000 Samples
------------------------------------------------------------


  Standard (Random)    - Acc: 0.9186 | Loss: 0.06105
  Hard Negative (Key)  - Acc: 0.9194 | Loss: 0.06212

Evaluating: finetuned_hn_dnd_speck32_r6_d10.pt
Configuration: 6 Rounds, Depth 10, 1000000 Samples
------------------------------------------------------------


  Standard (Random)    - Acc: 0.7856 | Loss: 0.14793
  Hard Negative (Key)  - Acc: 0.7780 | Loss: 0.15293

Evaluating: finetuned_hn_dnd_speck32_r7_d1.pt
Configuration: 7 Rounds, Depth 1, 1000000 Samples
------------------------------------------------------------


  Standard (Random)    - Acc: 0.5505 | Loss: 0.24575
  Hard Negative (Key)  - Acc: 0.5445 | Loss: 0.24657

Evaluating: finetuned_hn_dnd_speck32_r8_d1.pt
Configuration: 8 Rounds, Depth 1, 1000000 Samples
------------------------------------------------------------


  Standard (Random)    - Acc: 0.5012 | Loss: 0.25000
  Hard Negative (Key)  - Acc: 0.4997 | Loss: 0.25000
